# 1. Data and calculations

pemlab turns raw CSV files into numbers and LaTeX files for your report. This notebook covers the
first half: **load a CSV, calculate new columns, propagate uncertainties, write everything to the
build folder.** Plots and fits follow in the next notebook.

The data is `data/resistor.csv`: the voltage across an ohmic resistor at several currents. The
raw file is never changed.

In [1]:
import pandas as pd

pd.read_csv("data/resistor.csv").head(3)

,U [V],I [mA]
0,0.943,2.0
1,1.870,4.0
2,2.828,6.0


## The project

A `Project` knows the build folder. Set it once, and at the end a single `lab.build()` writes
everything there. `tex_prefix` is the path with which LaTeX finds the build folder from your
report.

In [2]:
from pemlab import NumberStyle, Project

lab = Project("pemlab_build", tex_prefix="pemlab_build")

## Load a CSV

`edit_csv` loads the file for editing. A column name is a label, by convention with the unit in
square brackets. pemlab derives nothing from the names, so write each name **once** as a variable
and reuse it. Names may contain math (`$\sigma_U$`) and `\Omega`.

In [3]:
U = "U [V]"
I_mA = "I [mA]"
I = "I [A]"
SIGMA_U = r"$\sigma_U$ [V]"
R = r"$R$ [\Omega]"
SIGMA_R = r"$\sigma_R$ [\Omega]"

resistor = lab.edit_csv("data/resistor.csv", name="resistor")
resistor.df.head(3)

,U [V],I [mA]
0,0.943,2.0
1,1.870,4.0
2,2.828,6.0


## Calculate columns

Every editing method returns the table, so the notebook shows its current state after each step.

**`replace_column`** changes a column in place. The formula has the variable `I_mA`, and the
keyword `I_mA=I_mA` assigns the old column to it (first the name in the formula, then our variable
with the column name). `show=False` keeps this trivial unit conversion out of the report.

In [4]:
resistor.replace_column(I_mA, I, "I_mA * 1e-3", I_mA=I_mA, show=False)

,U [V],I [A]
0,0.943,0.002
1,1.870,0.004
2,2.828,0.006
3,3.769,0.008
4,4.680,0.010
5,5.627,0.012
6,6.581,0.014
7,7.517,0.016
8,8.460,0.018
9,9.391,0.020


**`add_uncertainty`** propagates uncertainties (analytically, with the formula). `sigmas` names the
uncertainty of every variable. The voltmeter has no uncertainty column, so we pass its accuracy as
a constant; the formula `"U"` turns it into a column.

In [5]:
resistor.add_uncertainty(SIGMA_U, "U", {"U": 0.01}, U=U)

,U [V],I [A],$\sigma_U$ [V]
0,0.943,0.002,0.01
1,1.870,0.004,0.01
2,2.828,0.006,0.01
3,3.769,0.008,0.01
4,4.680,0.010,0.01
5,5.627,0.012,0.01
6,6.581,0.014,0.01
7,7.517,0.016,0.01
8,8.460,0.018,0.01
9,9.391,0.020,0.01


**`add_column`** adds a column from a formula. `NumberStyle(decimals=1)` rounds only the
**display** in the table; the stored values keep all digits. `show=True` lists the calculation in
the report. A variable that is the name of a column (`U` for `U [V]`) needs no assignment.

In [6]:
resistor.add_column(R, "U / I", U=U, I=I, numbers=NumberStyle(decimals=1), show=True)

,U [V],I [A],$\sigma_U$ [V],$R$ [\Omega]
0,0.943,0.002,0.01,471.500000
1,1.870,0.004,0.01,467.500000
2,2.828,0.006,0.01,471.333333
3,3.769,0.008,0.01,471.125000
4,4.680,0.010,0.01,468.000000
5,5.627,0.012,0.01,468.916667
6,6.581,0.014,0.01,470.071429
7,7.517,0.016,0.01,469.812500
8,8.460,0.018,0.01,470.000000
9,9.391,0.020,0.01,469.550000


The uncertainty of $R = U/I$: pemlab differentiates the formula. `U` has the column we just
created as uncertainty, `I` a constant.

In [7]:
resistor.add_uncertainty(
    SIGMA_R, "U / I", {"U": SIGMA_U, "I": 2e-5}, U=U, I=I,
    numbers=NumberStyle(decimals=1), show=True,
)

,U [V],I [A],$\sigma_U$ [V],$R$ [\Omega],$\sigma_R$ [\Omega]
0,0.943,0.002,0.01,471.500000,6.872498
1,1.870,0.004,0.01,467.500000,3.422558
2,2.828,0.006,0.01,471.333333,2.290451
3,3.769,0.008,0.01,471.125000,1.717481
4,4.680,0.010,0.01,468.000000,1.369707
5,5.627,0.012,0.01,468.916667,1.142467
6,6.581,0.014,0.01,470.071429,0.980386
7,7.517,0.016,0.01,469.812500,0.857616
8,8.460,0.018,0.01,470.000000,0.762468
9,9.391,0.020,0.01,469.550000,0.685913


**`set_column`** adds values you computed yourself, for example fit results. It is shown rounded
like the others, the CSV keeps all digits.

In [8]:
resistor.set_column("R/R_mean", resistor.column(R) / resistor.column(R).mean(),
                    numbers=NumberStyle(decimals=3))

,U [V],I [A],$\sigma_U$ [V],$R$ [\Omega],$\sigma_R$ [\Omega],R/R_mean
0,0.943,0.002,0.01,471.500000,6.872498,1.003659
1,1.870,0.004,0.01,467.500000,3.422558,0.995145
2,2.828,0.006,0.01,471.333333,2.290451,1.003305
3,3.769,0.008,0.01,471.125000,1.717481,1.002861
4,4.680,0.010,0.01,468.000000,1.369707,0.996209
5,5.627,0.012,0.01,468.916667,1.142467,0.998160
6,6.581,0.014,0.01,470.071429,0.980386,1.000618
7,7.517,0.016,0.01,469.812500,0.857616,1.000067
8,8.460,0.018,0.01,470.000000,0.762468,1.000466
9,9.391,0.020,0.01,469.550000,0.685913,0.999509


Every calculation is recorded. This is what ends up in the report:

In [9]:
for calculation in resistor.calculations:
    print(f"{calculation.column:>22}  =  {calculation.formula}")

                 I [A]  =  I_mA * 1e-3
        $\sigma_U$ [V]  =  U
          $R$ [\Omega]  =  U / I
   $\sigma_R$ [\Omega]  =  U / I
              R/R_mean  =  


## Tables and build

`tables_from_folder` creates a LaTeX table for every CSV in a folder. `build` writes everything
and deletes stale files first.

In [10]:
lab.tables_from_folder("data")
lab.build()

In [11]:
from pathlib import Path

build = Path("pemlab_build")
for path in sorted(build.rglob("*")):
    if path.is_file():
        print(path.relative_to(build).as_posix())

.pemlab-build
data/resistor/resistor.calc.json
data/resistor/resistor.csv
data/resistor/resistor.tex
tables/lowpass/lowpass.tex
tables/resistor/resistor.tex


The edited table carries its calculations in the same way:

In [12]:
print((build / "data" / "resistor" / "resistor.tex").read_text(encoding="utf-8"))

\begin{table}[H]
  \centering
  \begin{tabular}{@{}cccccc@{}}
    \toprule
    U [V] & I [A] & $\sigma_U$ [V] & $R$ [$\Omega$] & $\sigma_R$ [$\Omega$] & R/R\_mean \\
    \midrule
    0.943 & \num{2e-3} & 0.01000 & 471.5 & 6.9 & 1.004 \\
    1.87 & \num{4e-3} & 0.01000 & 467.5 & 3.4 & 0.995 \\
    2.828 & \num{6e-3} & 0.01000 & 471.3 & 2.3 & 1.003 \\
    3.769 & \num{8e-3} & 0.01000 & 471.1 & 1.7 & 1.003 \\
    4.68 & 0.01 & 0.01000 & 468.0 & 1.4 & 0.996 \\
    5.627 & 0.012 & 0.01000 & 468.9 & 1.1 & 0.998 \\
    6.581 & 0.014 & 0.01000 & 470.1 & 1.0 & 1.001 \\
    7.517 & 0.016 & 0.01000 & 469.8 & 0.9 & 1.000 \\
    8.46 & 0.018 & 0.01000 & 470.0 & 0.8 & 1.000 \\
    9.391 & 0.02 & 0.01000 & 469.6 & 0.7 & 1.000 \\
    \bottomrule
  \end{tabular}
  \caption{resistor}
  \label{tab:resistor}
  \par\medskip
  {\footnotesize
Calculations:
$R = \frac{U}{I}$
\qquad
$\sigma_R = \sqrt{\frac{\sigma_{U}^{2}}{I^{2}} + \frac{U^{2} \cdot \sigma_{I}^{2}}{I^{4}}}$ (with $\sigma_{I} = \SI{2e-05}{A}$)
 

## In your report

You need `graphicx`, `booktabs`, `siunitx` and `float`. Then include what you want, where you
want it:

```latex
\input{pemlab_build/data/resistor/resistor}
```

Next: plots and fits (notebook 2).